# 05 - Diagnostics, Failed-OOS Review, and Research Decision

Produce the Phase 1 evidence bundle, preserve the retired frozen-candidate result, and review historical recency research without tuning against the failed future-OOS window.

In [ ]:
import torch, sys
if torch.cuda.is_available():
    print(f"GPU available but diagnostics can run on CPU: {torch.cuda.get_device_name(0)}")
else:
    print("CPU diagnostics mode: no GPU required for notebook 05")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_BASE  = '/content/drive/MyDrive/yeniBot'
DATA_DIR    = f'{DRIVE_BASE}/data'
CHECKPT_DIR = f'{DRIVE_BASE}/checkpoints'
REPORT_DIR  = f'{DRIVE_BASE}/reports'
os.makedirs(CHECKPT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/umutergul74/yeniBot.git'
REPO_DIR = '/content/yenibot_repo'
REPO_BRANCH = os.environ.get('YENIBOT_REPO_BRANCH', 'research/next-candidate-v1')
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, REPO_DIR], check=True)
repo_commit = subprocess.check_output(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], text=True).strip()
repo_branch = subprocess.check_output(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True).strip()
assert repo_branch == REPO_BRANCH, f'Expected {REPO_BRANCH}, found {repo_branch}'
sys.path.insert(0, REPO_DIR)
print('Repository branch:', repo_branch)
print('Repository commit:', repo_commit)
print('After a changed checkout, use Runtime -> Restart session before trusting imports.')

In [ ]:
!pip install -q -r {REPO_DIR}/requirements.txt

In [ ]:
import yaml
with open(f'{REPO_DIR}/config.yaml') as f:
    cfg = yaml.safe_load(f)
cfg['paths']['data_dir'] = DATA_DIR
cfg['paths']['checkpoint_dir'] = CHECKPT_DIR
research = cfg.get('experiments', {}).get('next_research_cycle', {})
failed_outcomes = cfg.get('experiments', {}).get('frozen_candidate_outcomes', {})
assert research.get('same_window_selection_allowed') is False
assert research.get('new_future_oos_anchor_required') is True
print('Policy review status:', cfg.get('experiments', {}).get('policy_review', {}).get('status'))
print('Research cycle:', research.get('status'))
print('Primary hypothesis:', research.get('primary_hypothesis'))
print('Retired frozen candidates:', list(failed_outcomes))

In [ ]:
AUTO_UNASSIGN = True
WRITE_FULL_BUNDLE = False  # Slim bundle contains the required review evidence.
RUN_ID = None  # Set an explicit experiment run id only when reviewing an older run.

import json
import pandas as pd
from google.colab import runtime
from pathlib import Path
from IPython.display import Image, display as ipy_display
import yenibot.experiment.orchestration as experiments_module
from yenibot.experiment import future_oos_preflight, latest_experiment_run, write_experiment_diagnostics

def display_csv(report_dir, filename, max_rows=30):
    path = Path(report_dir) / filename
    if not path.exists():
        print('Missing report:', filename)
        return pd.DataFrame()
    try:
        frame = pd.read_csv(path)
    except pd.errors.EmptyDataError:
        print(filename, '- empty report (no rows or columns)')
        return pd.DataFrame()
    print(filename, '- rows:', len(frame))
    display(frame.head(max_rows))
    return frame

try:
    print('yenibot orchestration module:', experiments_module.__file__)
    handoff_path = Path(CHECKPT_DIR) / 'notebook04_run.json'
    if RUN_ID:
        selected_run_id = RUN_ID
        run_source = 'explicit_run_id'
    elif handoff_path.exists():
        handoff = json.loads(handoff_path.read_text(encoding='utf-8'))
        selected_run_id = str(handoff['run_id'])
        run_source = 'notebook04_handoff'
        if handoff.get('repo_branch'):
            assert handoff['repo_branch'] == repo_branch, 'Notebook 04 used a different repository branch'
        if research.get('recency_ensemble', {}).get('enabled', False):
            assert handoff.get('recency_research_completed') is True, 'Notebook 04 recency research did not complete'
        if research.get('replacement_candidate', {}).get('enabled', False):
            assert handoff.get('replacement_candidate_fit_completed') is True, 'Notebook 04 replacement candidate fit did not complete'
    else:
        selected_run_id = latest_experiment_run(CHECKPT_DIR).name
        run_source = 'latest_experiment_fallback'
    run_dir = Path(CHECKPT_DIR) / 'experiments' / selected_run_id
    if not run_dir.exists():
        raise FileNotFoundError(f'Experiment run does not exist: {run_dir}')
    print('Selected experiment run:', run_dir)
    print('Run selection source:', run_source)
    preflight = future_oos_preflight(checkpoint_dir=CHECKPT_DIR, config=cfg)
    print('Read-only future OOS preflight:', preflight['state'])
    print('Fresh labeled rows:', preflight['data']['fresh_labeled_rows'], '/', preflight['data']['min_rows'])
    print('Preflight failed checks:', preflight['failed_checks'])
    if cfg.get('experiments', {}).get('future_oos_validation', {}).get('enabled', False):
        print('Frozen candidate outcome is immutable; future-OOS diagnostics remain read-only.')
    else:
        print('Future-OOS scoring is paused while historical walk-forward CV is repaired.')
    print('Research focus:', cfg.get('experiments', {}).get('research_focus', {}))
    print('Same-window policy selection allowed:', research.get('same_window_selection_allowed'))
    result = write_experiment_diagnostics(
        checkpoint_dir=CHECKPT_DIR,
        config=cfg,
        output_dir=REPORT_DIR,
        run_id=run_dir.name,
        write_full_bundles=WRITE_FULL_BUNDLE,
    )
    display(result['comparison'])
    if not result.get('missing_selected_profiles').empty:
        print('Missing selected profiles detected:')
        display(result['missing_selected_profiles'])
    if not result.get('seed_stability').empty:
        display(result['seed_stability'])
    if not result.get('seed_audit_coverage').empty:
        print('Seed audit coverage:')
        display(result['seed_audit_coverage'])
    if not result.get('seed_ensemble').empty:
        display(result['seed_ensemble'])
    if not result.get('experiment_selection').empty:
        display(result['experiment_selection'])
    if not result.get('experiment_policy_guard').empty:
        print('Experiment policy guard:')
        display(result['experiment_policy_guard'])
    if not result.get('future_oos_candidate_plan').empty:
        print('Future OOS candidate plan:')
        display(result['future_oos_candidate_plan'])
    if not result.get('performance_gap_analysis').empty:
        print('Performance gap analysis:')
        display(result['performance_gap_analysis'])
    if not result.get('phase1_blocker_root_cause').empty:
        print('Phase 1 blocker root cause:')
        display(result['phase1_blocker_root_cause'])
    if not result.get('threshold_oracle_gap').empty:
        print('Threshold oracle gap:')
        display(result['threshold_oracle_gap'])
    if not result.get('bad_fold_mechanism_summary').empty:
        print('Bad-fold mechanism summary:')
        display(result['bad_fold_mechanism_summary'])
    if not result.get('score_reversal_context_audit').empty:
        print('Score reversal context audit:')
        display(result['score_reversal_context_audit'])
    if not result.get('rank_ic_variance_decomposition').empty:
        print('Rank IC variance decomposition:')
        display(result['rank_ic_variance_decomposition'])
    if not result.get('rank_ic_aggregate_evidence').empty:
        print('Rank IC aggregate evidence:')
        display(result['rank_ic_aggregate_evidence'])
    if not result.get('causal_threshold_policy_summary').empty:
        print('Causal threshold policy summary:')
        display(result['causal_threshold_policy_summary'])
    if not result.get('classification_skill_summary').empty:
        print('Classification skill summary:')
        display(result['classification_skill_summary'])
    if not result.get('validation_charter_review').empty:
        print('Validation charter review:')
        display(result['validation_charter_review'])
    if not result.get('validation_charter_proposal').empty:
        print('Active validation charter evidence:')
        display(result['validation_charter_proposal'])
    if not result.get('validation_charter_status').empty:
        print('Validation charter governance status:')
        display(result['validation_charter_status'])
    if not result.get('frozen_candidate_index').empty:
        print('Frozen candidate artifact index:')
        display(result['frozen_candidate_index'])
    print('Future OOS readiness:', result.get('future_oos_readiness'))
    if not result.get('future_oos_evaluation').empty:
        print('Future OOS evaluation:')
        display(result['future_oos_evaluation'])
    report_dir = Path(result['report_dir'])
    display_csv(report_dir, 'preprocessing_audit_summary.csv', max_rows=20)
    display_csv(report_dir, 'preprocessing_audit.csv', max_rows=60)
    failure_path = report_dir / 'future_oos_failure_summary.json'
    if failure_path.exists():
        failure_summary = json.loads(failure_path.read_text(encoding='utf-8'))
        print('Future OOS failure mechanism:')
        print(json.dumps(failure_summary, indent=2))
    display_csv(report_dir, 'future_oos_temporal_blocks.csv')
    display_csv(report_dir, 'future_oos_score_bands.csv')
    display_csv(report_dir, 'future_oos_regime_metrics.csv')
    display_csv(report_dir, 'future_oos_ensemble_disagreement.csv')
    display_csv(report_dir, 'future_oos_model_metrics.csv')
    recency_summary = display_csv(report_dir, 'recency_ensemble_summary.csv')
    display_csv(report_dir, 'recency_ensemble_by_fold.csv')
    display_csv(report_dir, 'recency_ensemble_schedule.csv', max_rows=10)
    recency_audit = display_csv(report_dir, 'recency_ensemble_eligibility_audit.csv', max_rows=10)
    if not recency_audit.empty:
        assert recency_audit['eligible'].all(), 'Recency eligibility audit failed'
    display_csv(report_dir, 'recency_ensemble_paired_comparison.csv')
    recency_decision_path = report_dir / 'recency_ensemble_decision.json'
    if recency_decision_path.exists():
        print('Recency ensemble decision:')
        print(json.dumps(json.loads(recency_decision_path.read_text(encoding='utf-8')), indent=2))
    replacement_path = report_dir / 'replacement_candidate_fit.json'
    if replacement_path.exists():
        print('Replacement candidate fit:')
        print(json.dumps(json.loads(replacement_path.read_text(encoding='utf-8')), indent=2))
    protocol_path = report_dir / 'next_research_protocol.json'
    if protocol_path.exists():
        protocol = json.loads(protocol_path.read_text(encoding='utf-8'))
        print('Next research protocol:')
        print(json.dumps(protocol, indent=2))
    if not result.get('payoff_alignment_summary').empty:
        print('Payoff alignment summary:')
        display(result['payoff_alignment_summary'])
    if not result.get('payoff_policy_robustness_summary').empty:
        print('Payoff policy robustness summary:')
        display(result['payoff_policy_robustness_summary'])
    if not result.get('model_performance_scorecard').empty:
        print('Model performance scorecard:')
        display(result['model_performance_scorecard'])
        print('Executive model dashboard:')
        dashboard_dir = report_dir
        for image_name in [
            'model_scorecard.png',
            'rank_ic_stability.png',
            'classification_quality.png',
            'score_band_payoff.png',
        ]:
            image_path = dashboard_dir / image_name
            if image_path.exists():
                ipy_display(Image(filename=str(image_path)))
        print('Dashboard tables and visuals are included in the slim bundle.')
    if not result.get('frozen_policy_monitoring_plan').empty:
        print('Frozen policy monitoring plan:')
        display(result['frozen_policy_monitoring_plan'])
    if not result.get('holdout_evaluation').empty:
        print('Holdout evaluation:')
        display(result['holdout_evaluation'])
    if not result.get('holdout_score_bands').empty:
        print('Holdout score bands:')
        display(result['holdout_score_bands'])
    holdout = result['decision'].get('holdout', {})
    if holdout:
        print('Holdout reservation:', holdout)
    print('Decision:', result['decision'])
    print('Full bundle enabled:', result.get('write_full_bundles'))
    print('Diagnostic zips:')
    for zip_path in result['zip_paths']:
        print(zip_path)
    print('Single download bundle:', result['bundle_zip'])
    print('Latest bundle shortcut:', result['latest_bundle_zip'])
    print('Slim summary bundle:', result.get('slim_bundle_zip'))
    print('Latest slim shortcut:', result.get('latest_slim_bundle_zip'))
    print('Auto review:', result.get('auto_review_path'))
    print('Next actions:', result.get('next_actions_path'))
    print('Phase 2 readiness:', result.get('phase2_readiness_path'))
finally:
    if AUTO_UNASSIGN:
        runtime.unassign()
